# Pixel3DMM — Milestone 1 Geometry Bake-Off (Colab H100)

Hair App 첫 3D 실험: 사용자 다중 사진에서 **hairless head mesh**를 재현한다.

- 계획: `experiments/milestone1_geometry_bakeoff/README.md`
- source of truth: `docs/10_3d_hair_app_master_plan.md` (Milestone 1)
- **License: CC BY-NC 4.0 (비상업 연구). 상용 가능 아님.**

> ⚠️ **Privacy:** private 사진/출력은 절대 git에 넣지 않는다. Drive에만 저장한다.
> ⚠️ 아래 명령은 2026-06-21 공식 README 기준이다. Colab/CUDA 변화로 깨질 수 있으니,
> 통과한 exact version과 fix를 manifest에 기록한다 (docs/07 방식).

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. conda 설치 (condacolab)

Pixel3DMM은 conda 기반이다. Colab에는 conda가 없으므로 `condacolab`을 설치한다.
이 셀 실행 후 **커널이 자동 재시작**된다. 재시작되면 다음 셀부터 이어서 실행한다.

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()  # 커널이 재시작됨 (정상)

## 2. 저장소 clone

In [ ]:
import condacolab; condacolab.check()
import os
%cd /content
if not os.path.exists('/content/pixel3dmm'):
    !git clone https://github.com/SimonGiebenhain/pixel3dmm.git
%cd /content/pixel3dmm
# 재현성을 위해 commit 고정 기록
!git rev-parse HEAD

## 3. 환경 생성

공식 README의 manual 경로. `environment.yml`이 Colab에서 충돌하면 아래 manual 설치를 사용한다.

⚠️ **가장 깨지기 쉬운 단계** = pytorch3d / nvdiffrast 빌드.
`TORCH_CUDA_ARCH_LIST`를 Colab GPU에 맞춰야 할 수 있다 (H100 = `9.0`).
실패하면 에러와 통과한 수정을 manifest에 남긴다.

In [ ]:
%%bash
set -e
source activate base || true
conda create -n p3dmm python=3.9 -y
# 이후 셀은 `conda run -n p3dmm ...` 로 실행한다 (Colab은 conda activate 지속이 어려움)

In [ ]:
%%bash
set -e
# torch (cu118) + 빌드 의존성
conda run -n p3dmm pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# H100 기준 arch. 다른 GPU면 수정.
conda run -n p3dmm conda env config vars set TORCH_CUDA_ARCH_LIST="9.0+PTX"
conda run -n p3dmm pip install git+https://github.com/facebookresearch/pytorch3d.git@stable
conda run -n p3dmm pip install git+https://github.com/NVlabs/nvdiffrast.git
cd /content/pixel3dmm
conda run -n p3dmm pip install -r requirements.txt
conda run -n p3dmm pip install -e .

## 4. 전처리 파이프라인 + FLAME 자산 다운로드

`download_flame2023.sh`는 https://flame.is.tue.mpg.de 계정/동의가 필요할 수 있다.

In [ ]:
%%bash
set -e
cd /content/pixel3dmm
conda run -n p3dmm ./install_preprocessing_pipeline.sh
conda run -n p3dmm ./download_flame2023.sh   # FLAME 등록 필요할 수 있음

## 5. 환경 변수 파일 작성

`~/.config/pixel3dmm/.env` 에 경로를 지정한다. 출력은 Drive 등 persistent 위치로.

In [ ]:
import os, pathlib
cfg = pathlib.Path.home() / '.config' / 'pixel3dmm'
cfg.mkdir(parents=True, exist_ok=True)
env = (
    'PIXEL3DMM_CODE_BASE="/content/pixel3dmm"\n'
    'PIXEL3DMM_PREPROCESSED_DATA="/content/p3dmm_preprocessed"\n'
    'PIXEL3DMM_TRACKING_OUTPUT="/content/p3dmm_tracking"\n'
)
(cfg / '.env').write_text(env)
print((cfg / '.env').read_text())

## 6. private 입력 준비 (git 금지)

Drive를 마운트하고 본인 사진 폴더를 가리킨다. README의 Input Checklist 참고.
Pixel3DMM은 video 또는 image 폴더를 받는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# TODO: 본인 입력 세트 경로로 교체
INPUT_PATH = '/content/drive/MyDrive/hair_app_private/m1_inputs/set01'
VID_NAME = 'set01'   # 결과 폴더 식별자
import os; print('exists:', os.path.exists(INPUT_PATH))

## 7. Step 1 — 전처리 (crop/landmark/mask)

In [ ]:
%%bash -s "$INPUT_PATH"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/run_preprocessing.py --video_or_images_path "$1"

## 8. Step 2 — 네트워크 추론 (normals, uv_map)

In [ ]:
%%bash -s "$VID_NAME"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=normals video_name="$1"
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=uv_map video_name="$1"

## 9. Step 3 — 트래킹 (multi-image 모드)

다중 사진 identity fusion 설정. 단일 이미지면 `python scripts/track.py video_name=$VID_NAME iters=800`.

In [ ]:
%%bash -s "$VID_NAME"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/track.py video_name="$1" \
  iters=100 iters=1500 include_neck=False use_flame2023=True ignore_mica=True is_discontinuous=True

## 10. 출력 확인 / 렌더

tracking output(mesh, FLAME params, cameras)을 찾아 같은 camera·neutral material로 렌더한다.
KaoLRM과 **동일한 view 세트**(정면/좌우 3/4/profile/후면)로 비교 렌더를 만든다.

In [ ]:
import glob
outs = glob.glob('/content/p3dmm_tracking/**/*', recursive=True)
for p in sorted(outs)[:50]:
    print(p)
# TODO: mesh를 동일 camera/gray material로 렌더 (pyrender/PyTorch3D)하여 scoring 이미지 생성

## 11. Run manifest 기록 (재현성)

In [ ]:
import json, subprocess, datetime, pathlib
commit = subprocess.run(['git','-C','/content/pixel3dmm','rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
manifest = {
    'model': 'pixel3dmm',
    'commit': commit,
    'license': 'CC BY-NC 4.0 (non-commercial)',
    'input_set_id': VID_NAME,
    'tracking_config': 'iters=100 iters=1500 include_neck=False use_flame2023=True ignore_mica=True is_discontinuous=True',
    'gpu': 'H100',
    'created_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'fixes_applied': [],   # TODO: 실제로 적용한 수정 기록
}
# private 위치(Drive)에 저장. git 금지.
dst = pathlib.Path('/content/drive/MyDrive/hair_app_private/m1_manifests')
dst.mkdir(parents=True, exist_ok=True)
out = dst / f'pixel3dmm_{VID_NAME}.json'
out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('saved', out)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 12. 점수화

렌더를 보고 `scoring_sheet.csv`에 identity/geometry/hairline/side_contour/scalp_ear_topology/
execution_reliability(1–5)를 기록한다. hidden scalp/rear는 측정값이 아니라 prior 추정임을 유의.